# Taller de Procesamiento de Datos — VENTAS_NL

**Machine Learning — Universidad Libre**

## Objetivo

Aplicar un flujo completo de procesamiento de datos — limpieza, estandarización e ingeniería de características — sobre la base de datos VENTAS_NL, y a partir de ese análisis, proponer y justificar un modelo de predicción.

> **Nota sobre los datos:** el archivo `data_source.xlsx` incluido en este repositorio es equivalente al `VENTAS_NL.xlsx` descrito en el enunciado del taller.

## Importación de librerías

Importamos las dependencias necesarias para la carga y exploración de datos, visualización, preprocesamiento con scikit-learn y evaluación de modelos. Estas librerías se reutilizarán a lo largo de las secciones de limpieza, estandarización, ingeniería de características y modelado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## Carga de datos y renombrado de columnas

Leemos el archivo Excel y renombramos la columna `costo venta` a `costo_venta` para facilitar el acceso en Python (sin espacios) y mantener consistencia con el resto del notebook. Verificamos la forma del DataFrame y que el rename se aplicó correctamente.

In [ ]:
df = pd.read_excel("data_source.xlsx")
df = df.rename(columns={"costo venta": "costo_venta"})
print(df.shape)
print(df.columns.tolist())
assert "costo_venta" in df.columns
assert "costo venta" not in df.columns
assert df.shape[1] == 8

## Exploración inicial — tipos de datos

Antes de limpiar o transformar el dataset, revisamos los tipos inferidos por pandas en cada columna. Esto nos permite detectar columnas numéricas leídas como texto, fechas mal interpretadas u otros problemas de tipado que afecten el análisis posterior.

In [ ]:
print(df.dtypes)

## Exploración inicial — valores faltantes

Inventariamos los valores nulos por columna para priorizar la limpieza: identificamos qué variables tienen faltantes, en qué magnitud y si conviene imputar, eliminar filas o tratarlos en una etapa posterior.

In [ ]:
nulos = df.isna().sum()
print(nulos)
print("Total nulos:", int(nulos.sum()))

## Exploración inicial — rangos y categorías

Con `describe()` obtenemos estadísticas descriptivas de las variables numéricas y un resumen de las categóricas. Complementamos con el conteo de `GENERO`, los valores únicos de `SIZE` y el rango de `EDAD` para entender la distribución y detectar valores atípicos o categorías inesperadas.

In [ ]:
print(df.describe(include="all"))
print("GENERO:", df["GENERO"].value_counts(dropna=False).to_dict())
print("SIZE unique:", sorted(df["SIZE"].dropna().unique().tolist()))
print("EDAD min/max:", df["EDAD"].min(), df["EDAD"].max())

## Duplicados por `id`

Identificamos filas con el mismo identificador (`id`). Estos duplicados globales se eliminan conservando la primera aparición de cada `id`, ya que representan registros repetidos que no deben contarse dos veces en el análisis.

In [ ]:
n_before = len(df)
n_dup_id = int(df.duplicated(subset=["id"]).sum())
print("Duplicados por id:", n_dup_id)
df = df.drop_duplicates(subset=["id"], keep="first").copy()
n_after = len(df)
print("Filas antes:", n_before, "después:", n_after, "eliminadas:", n_before - n_after)
assert df["id"].is_unique
assert n_before - n_after == n_dup_id

## Duplicados estructurales

Comparamos todas las columnas excepto `id` para detectar filas idénticas en contenido. Estas filas se marcan con la columna booleana `dup_estructura` (todas las filas del grupo quedan marcadas con `keep=False`), pero **no se eliminan** del dataset.

In [ ]:
cols_estructura = [c for c in df.columns if c != "id"]
df["dup_estructura"] = df.duplicated(subset=cols_estructura, keep=False)
print("Filas con dup_estructura=True:", int(df["dup_estructura"].sum()))
print("Grupos estructurales duplicados (keep='first' count):", int(df.duplicated(subset=cols_estructura).sum()))
assert "dup_estructura" in df.columns
assert df["dup_estructura"].dtype == bool

## Duplicados significativos

Detectamos filas que coinciden en el perfil demográfico y económico (`EDAD`, `GENERO`, `SIZE`, `YEARINCOME`). Se marcan con `dup_significativo` sin eliminar filas, para analizar posibles registros redundantes o patrones repetidos en esas variables.

In [ ]:
cols_sig = ["EDAD", "GENERO", "SIZE", "YEARINCOME"]
df["dup_significativo"] = df.duplicated(subset=cols_sig, keep=False)
print("Filas con dup_significativo=True:", int(df["dup_significativo"].sum()))
assert set(cols_sig).issubset(df.columns)

## Limpieza

A partir de esta sección aplicamos transformaciones sobre `df` ya depurado de duplicados por `id`: estandarización de tipos, validación de rangos y reglas de coherencia de negocio.

### Formatos — coerción de tipos numéricos

Convertimos las columnas numéricas con `pd.to_numeric(..., errors="coerce")` para detectar valores no parseables como nulos, y normalizamos `GENERO` a texto en mayúsculas sin espacios laterales.

In [ ]:
num_cols = ["EDAD", "SIZE", "YEARINCOME", "ventas", "costo_venta", "DESCUENTOS"]
for c in num_cols:
    before_na = df[c].isna().sum()
    df[c] = pd.to_numeric(df[c], errors="coerce")
    after_na = df[c].isna().sum()
    print(f"{c}: nulos {before_na} -> {after_na}")
df["GENERO"] = df["GENERO"].astype(str).str.strip().str.upper()
print("GENERO values:", df["GENERO"].value_counts(dropna=False).to_dict())

### Rangos — validación categórica y numérica

Verificamos que `SIZE` esté en el rango 1–5 y `EDAD` entre 18 y 100. Imprimimos conteos de valores fuera de rango y afirmamos que no queden casos inválidos en los datos actuales.

In [ ]:
print("SIZE fuera de 1..5:", int((~df["SIZE"].between(1, 5) & df["SIZE"].notna()).sum()))
print("EDAD fuera 18..100:", int((~df["EDAD"].between(18, 100) & df["EDAD"].notna()).sum()))
assert df["SIZE"].dropna().between(1, 5).all()
assert df["EDAD"].dropna().between(18, 100).all()

### Coherencia — clip seguro y flags de negocio

Detectamos valores negativos y descuentos mayores a 1; los recortamos a 0 o 1 según corresponda. Cuando `costo_venta` supera `ventas` solo marcamos con `flag_costo_gt_ventas` sin modificar los montos.

In [ ]:
df["flag_neg_YEARINCOME"] = df["YEARINCOME"].lt(0).fillna(False)
df["flag_neg_ventas"] = df["ventas"].lt(0).fillna(False)
df["flag_neg_costo_venta"] = df["costo_venta"].lt(0).fillna(False)
df["flag_neg_DESCUENTOS"] = df["DESCUENTOS"].lt(0).fillna(False)
df["flag_DESCUENTOS_gt_1"] = df["DESCUENTOS"].gt(1).fillna(False)
df["flag_costo_gt_ventas"] = (
    df["costo_venta"].notna() & df["ventas"].notna() & (df["costo_venta"] > df["ventas"])
)

df.loc[df["flag_neg_YEARINCOME"], "YEARINCOME"] = 0
df.loc[df["flag_neg_ventas"], "ventas"] = 0
df.loc[df["flag_neg_costo_venta"], "costo_venta"] = 0
df.loc[df["flag_neg_DESCUENTOS"], "DESCUENTOS"] = 0
df.loc[df["flag_DESCUENTOS_gt_1"], "DESCUENTOS"] = 1

flag_cols = [
    "flag_neg_YEARINCOME", "flag_neg_ventas", "flag_neg_costo_venta",
    "flag_neg_DESCUENTOS", "flag_DESCUENTOS_gt_1", "flag_costo_gt_ventas",
]
print(df[flag_cols].sum())
assert (df["YEARINCOME"].dropna() >= 0).all()
assert (df["ventas"].dropna() >= 0).all()
assert (df["costo_venta"].dropna() >= 0).all()
assert (df["DESCUENTOS"].dropna().between(0, 1)).all()

## Asimetría e imputación — criterio de decisión

Para imputar valores faltantes en `YEARINCOME`, `ventas`, `costo_venta` y `DESCUENTOS`, medimos la asimetría (skewness) de cada columna sobre los valores no nulos y elegimos el estadístico de reemplazo según esta regla:

- **|skew| < 0.5** → imputar con la **media** (distribución aproximadamente simétrica).
- **|skew| ≥ 0.5** → imputar con la **mediana** (distribución sesgada; la mediana es más robusta ante colas largas).

La regresión solo se consideraría si existiera una correlación fuerte y justificable con otra variable predictora; en este caso, media o mediana según skewness es suficiente.

**No se eliminan filas** por valores nulos: solo se reemplazan los faltantes en las cuatro columnas indicadas.

In [ ]:
cols_na = ["YEARINCOME", "ventas", "costo_venta", "DESCUENTOS"]
filas_antes = len(df)
imputacion_resumen = []
for c in cols_na:
    skew = float(df[c].skew(skipna=True))
    metodo = "media" if abs(skew) < 0.5 else "mediana"
    imputacion_resumen.append({"columna": c, "skew": skew, "metodo": metodo, "nulos": int(df[c].isna().sum())})
resumen_df = pd.DataFrame(imputacion_resumen)
print(resumen_df)

### Justificación de métodos observados

Tras calcular la asimetría sobre los valores no nulos de cada columna, se aplicó la regla |skew| < 0.5 → media; |skew| ≥ 0.5 → mediana:

| Columna | Skew | Método | Nulos |
|---------|------|--------|-------|
| YEARINCOME | 1.27 | mediana | 1026 |
| ventas | 2.27 | mediana | 2528 |
| costo_venta | 2.33 | mediana | 1660 |
| DESCUENTOS | -0.41 | media | 1072 |

- **YEARINCOME** (|skew| = 1.27 ≥ 0.5): distribución sesgada positivamente; la mediana evita que ingresos extremos eleven el valor imputado.
- **ventas** (|skew| = 2.27 ≥ 0.5): cola derecha pronunciada; la mediana es más representativa que la media.
- **costo_venta** (|skew| = 2.33 ≥ 0.5): mismo patrón que ventas; se usa mediana por robustez ante outliers.
- **DESCUENTOS** (|skew| = 0.41 < 0.5): distribución casi simétrica; la media es adecuada.

No se requiere imputación por regresión: las correlaciones no justifican un modelo predictivo frente a media/mediana según skewness.

In [ ]:
for row in imputacion_resumen:
    c, metodo = row["columna"], row["metodo"]
    if metodo == "media":
        valor = df[c].mean()
    else:
        valor = df[c].median()
    df[c] = df[c].fillna(valor)
    print(f"Imputado {c} con {metodo}={valor}")

assert len(df) == filas_antes
assert df[cols_na].isna().sum().sum() == 0
assert (df["YEARINCOME"] >= 0).all()
assert (df["ventas"] >= 0).all()
assert (df["costo_venta"] >= 0).all()
assert df["DESCUENTOS"].between(0, 1).all()
print("Imputación OK; filas:", len(df))

## Estandarización — StandardScaler vs MinMaxScaler

Según el enunciado del taller, aplicamos dos tipos de escalamiento sobre el dataset ya limpio e imputado:

| Variable | Método | Columna nueva |
| --- | --- | --- |
| `YEARINCOME` | **StandardScaler** (escalamiento normal) | `YEARINCOME_std` |
| `ventas` | **StandardScaler** | `ventas_std` |
| `costo_venta` | **MinMaxScaler** (escala uniforme [0, 1]) | `costo_venta_minmax` |

**StandardScaler** resta la media y divide entre la desviación estándar: cada valor pasa a ser un **z-score** (desviaciones respecto a la media). Tras el ajuste, la media ≈ 0 y la desviación estándar ≈ 1. Un z-score de +1.5 indica que el registro está 1.5 desviaciones por encima del ingreso o venta promedio; −0.8, por debajo.

**MinMaxScaler** comprime los valores al intervalo **[0, 1]** usando el mínimo y máximo observados: 0 corresponde al costo mínimo y 1 al máximo. Facilita comparar magnitudes en una escala común cuando el rango absoluto importa más que la forma de la distribución.

Las columnas originales (`YEARINCOME`, `ventas`, `costo_venta`) se conservan para trazabilidad y análisis en unidades originales.

In [ ]:
scaler_std = StandardScaler()
df[["YEARINCOME_std", "ventas_std"]] = scaler_std.fit_transform(df[["YEARINCOME", "ventas"]])
print(df[["YEARINCOME_std", "ventas_std"]].describe())
assert abs(df["YEARINCOME_std"].mean()) < 1e-6
assert abs(df["ventas_std"].mean()) < 1e-6
assert abs(df["YEARINCOME_std"].std(ddof=0) - 1) < 1e-6
assert abs(df["ventas_std"].std(ddof=0) - 1) < 1e-6

### Normalización de `costo_venta` con MinMaxScaler

Para `costo_venta` el enunciado pide un **escalamiento uniforme entre 0 y 1**. MinMaxScaler transforma cada valor como `(x − min) / (max − min)`, de modo que el costo mínimo del dataset queda en 0 y el máximo en 1; los demás quedan proporcionalmente en ese intervalo.

Esta escala es útil cuando se desea interpretar un costo relativo al rango observado (p. ej. 0.75 ≈ tres cuartos del camino entre el costo mínimo y el máximo). La columna original `costo_venta` se mantiene sin modificar.

In [ ]:
scaler_mm = MinMaxScaler()
df["costo_venta_minmax"] = scaler_mm.fit_transform(df[["costo_venta"]])
print(df["costo_venta_minmax"].describe())
assert abs(df["costo_venta_minmax"].min() - 0) < 1e-9
assert abs(df["costo_venta_minmax"].max() - 1) < 1e-9
assert "costo_venta" in df.columns  # original retained